# Submission 2 v2 — LightGBM with Depth-1 Aggregations + Stability Filtering

Memory-optimized version (fixes OOM from v1).

Aggregates 4 depth-1 tables:
- `applprev_1` — previous loan applications
- `credit_bureau_a_1` — credit bureau A records
- `credit_bureau_b_1` — credit bureau B records
- `person_1` — person-level demographics

Memory strategy:
- Polars **lazy evaluation**: tables are never fully loaded, aggregation is streamed
- All numeric features cast to **float32** (halves RAM vs float64)
- Intermediate DataFrames deleted + `gc.collect()` after each join

Key addition: **stability-first feature filtering** — drops the 10% of features
with highest temporal drift to reduce the public→private score gap.

In [ ]:
import glob
import polars as pl
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from scipy.stats import linregress
import matplotlib.pyplot as plt
import gc
import os
import warnings
warnings.filterwarnings('ignore')

DATA = '/kaggle/input/home-credit-credit-risk-model-stability/parquet_files'
OUTPUT = '/kaggle/working'
TRAIN_CUTOFF = 60

# Verify data path
print(os.listdir(f'{DATA}/train')[:10])

## 1. Aggregate depth-1 tables (lazy — low memory)

`scan_parquet` + lazy `group_by` + `collect` streams through files
without ever loading them fully into RAM.

In [ ]:
def aggregate_lazy(pattern: str, name: str) -> pl.DataFrame:
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f'No files found: {pattern}')
    print(f'{name}: {len(files)} file(s)')

    # Determine target schema from first non-trivial file
    ref_schema = None
    for f in files:
        s = pl.read_parquet(f, n_rows=10).schema
        if any(v != pl.Null for v in s.values()):
            ref_schema = s
            break

    parts = []
    for f in files:
        part = pl.read_parquet(f)
        exprs = []
        for c in part.columns:
            if c == 'case_id':
                continue
            dtype = part[c].dtype
            if dtype == pl.Null and ref_schema and c in ref_schema:
                target = ref_schema[c]
                exprs.append(pl.col(c).cast(target if target != pl.Null else pl.Utf8))
            elif dtype in [pl.Float64, pl.Int64, pl.Int32, pl.Int16, pl.Int8]:
                exprs.append(pl.col(c).cast(pl.Float32))
        if exprs:
            part = part.with_columns(exprs)
        parts.append(part)

    df = pl.concat(parts, how='diagonal_relaxed')
    del parts
    gc.collect()

    schema = df.schema
    num_cols = [c for c, dtype in schema.items() if c != 'case_id' and dtype == pl.Float32]
    cat_cols = [c for c, dtype in schema.items() if c != 'case_id' and dtype == pl.Utf8]

    aggs = [pl.len().alias(f'{name}_count').cast(pl.Int32)]
    for col in num_cols:
        aggs += [
            pl.col(col).mean().alias(f'{name}_{col}_mean'),
            pl.col(col).max().alias(f'{name}_{col}_max'),
            pl.col(col).min().alias(f'{name}_{col}_min'),
            pl.col(col).std().alias(f'{name}_{col}_std'),
        ]
    for col in cat_cols:
        aggs.append(pl.col(col).n_unique().cast(pl.Int16).alias(f'{name}_{col}_nunique'))

    result = df.group_by('case_id').agg(aggs)
    del df
    gc.collect()
    print(f'  -> {result.shape}')
    return result

In [ ]:
agg_applprev      = aggregate_lazy(f'{DATA}/train/train_applprev_1_*.parquet',        'applprev')
agg_cb_a          = aggregate_lazy(f'{DATA}/train/train_credit_bureau_a_1_*.parquet', 'cb_a')
agg_cb_b          = aggregate_lazy(f'{DATA}/train/train_credit_bureau_b_1.parquet',   'cb_b')
agg_person        = aggregate_lazy(f'{DATA}/train/train_person_1.parquet',            'person')

agg_applprev_test = aggregate_lazy(f'{DATA}/test/test_applprev_1_*.parquet',          'applprev')
agg_cb_a_test     = aggregate_lazy(f'{DATA}/test/test_credit_bureau_a_1_*.parquet',   'cb_a')
agg_cb_b_test     = aggregate_lazy(f'{DATA}/test/test_credit_bureau_b_1.parquet',     'cb_b')
agg_person_test   = aggregate_lazy(f'{DATA}/test/test_person_1.parquet',              'person')

## 2. Load depth-0 tables and join everything

In [ ]:
def cast_static(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns([
        pl.col(c).cast(pl.Float32)
        for c in df.columns
        if df[c].dtype in [pl.Float64, pl.Int64, pl.Int32]
        and c not in ('case_id', 'target', 'WEEK_NUM', 'MONTH')
    ])

base      = pl.read_parquet(f'{DATA}/train/train_base.parquet')
static0   = pl.scan_parquet(sorted(glob.glob(f'{DATA}/train/train_static_0_*.parquet'))).collect().pipe(cast_static)
static_cb = pl.read_parquet(f'{DATA}/train/train_static_cb_0.parquet').pipe(cast_static)

test_base      = pl.read_parquet(f'{DATA}/test/test_base.parquet')
test_static0   = pl.scan_parquet(sorted(glob.glob(f'{DATA}/test/test_static_0_*.parquet'))).collect().pipe(cast_static)
test_static_cb = pl.read_parquet(f'{DATA}/test/test_static_cb_0.parquet').pipe(cast_static)

def build(base_df, s0, scb, aggs):
    df = base_df.join(s0, on='case_id', how='left').join(scb, on='case_id', how='left')
    for agg in aggs:
        df = df.join(agg, on='case_id', how='left')
    return df

df   = build(base, static0, static_cb, [agg_applprev, agg_cb_a, agg_cb_b, agg_person])
test = build(test_base, test_static0, test_static_cb, [agg_applprev_test, agg_cb_a_test, agg_cb_b_test, agg_person_test])

del static0, static_cb, test_static0, test_static_cb
del agg_applprev, agg_cb_a, agg_cb_b, agg_person
del agg_applprev_test, agg_cb_a_test, agg_cb_b_test, agg_person_test
gc.collect()

print(f'Train: {df.shape}')
print(f'Test:  {test.shape}')

## 3. Preprocessing

In [ ]:
DROP_COLS = ['case_id', 'date_decision', 'MONTH', 'WEEK_NUM', 'target']
TEST_DROP = ['case_id', 'date_decision', 'MONTH', 'WEEK_NUM']

def preprocess(df: pl.DataFrame, drop_cols: list) -> pl.DataFrame:
    date_cols = [c for c in df.columns if c.endswith('_D') and c not in drop_cols]
    for col in date_cols:
        df = df.with_columns(
            pl.col(col).str.to_date(strict=False).cast(pl.Int32).alias(col)
        )
    cat_cols = [c for c in df.columns if df[c].dtype == pl.Utf8 and c not in drop_cols]
    for col in cat_cols:
        df = df.with_columns(
            pl.col(col).cast(pl.Categorical).to_physical().cast(pl.Int16).alias(col)
        )
    return df.drop([c for c in drop_cols if c in df.columns])

week_num = df['WEEK_NUM'].to_numpy()
y = df['target'].to_numpy()

X_df      = preprocess(df, DROP_COLS)
X_test_df = preprocess(test, TEST_DROP)
del df, test
gc.collect()

feature_cols = [c for c in X_df.columns if c in X_test_df.columns]
X_df      = X_df.select(feature_cols)
X_test_df = X_test_df.select(feature_cols)

print(f'Features before stability filter: {len(feature_cols)}')

## 4. Stability-first feature filtering

For each numeric feature, compute its mean per WEEK_NUM on the training portion.
Features whose per-week mean drifts significantly (high coefficient of variation)
are likely to cause week-to-week instability — we drop the worst 10%.

In [ ]:
train_mask = week_num <= TRAIN_CUTOFF

numeric_cols = [
    c for c in X_df.columns
    if X_df[c].dtype in [pl.Float32, pl.Float64, pl.Int32, pl.Int16, pl.Int8]
]

df_wk = X_df.filter(pl.Series(train_mask)).select(numeric_cols).with_columns(
    pl.Series('WEEK_NUM', week_num[train_mask])
)
weekly_means = df_wk.group_by('WEEK_NUM').agg([pl.col(c).mean() for c in numeric_cols]).sort('WEEK_NUM')

drift = {}
for col in numeric_cols:
    vals = weekly_means[col].drop_nulls().to_numpy()
    if len(vals) < 5 or np.abs(vals.mean()) < 1e-9:
        drift[col] = 0.0
    else:
        drift[col] = vals.std() / np.abs(vals.mean())

del df_wk, weekly_means
gc.collect()

drift_df = pl.DataFrame({'feature': list(drift.keys()), 'drift': list(drift.values())}).sort('drift', descending=True)
n_drop = max(1, int(len(drift_df) * 0.10))
unstable = set(drift_df.head(n_drop)['feature'].to_list())
stable_features = [c for c in feature_cols if c not in unstable]

print(f'Dropped {len(unstable)} unstable features, {len(stable_features)} remaining')
print('Top 10 most unstable (dropped):')
print(drift_df.head(10))

In [ ]:
X      = X_df.select(stable_features).to_numpy()
X_test = X_test_df.select(stable_features).to_numpy()
del X_df, X_test_df
gc.collect()
print(f'Feature matrix: {X.shape}, Test: {X_test.shape}')

## 5. Gini stability metric

In [ ]:
def gini_stability(week_nums, targets, preds):
    weeks = np.unique(week_nums)
    weekly_gini = []
    for w in weeks:
        mask = week_nums == w
        if targets[mask].sum() == 0:
            continue
        weekly_gini.append(2 * roc_auc_score(targets[mask], preds[mask]) - 1)
    weekly_gini = np.array(weekly_gini)
    mean_gini = weekly_gini.mean()
    slope, intercept, *_ = linregress(np.arange(len(weekly_gini)), weekly_gini)
    residuals = weekly_gini - (intercept + slope * np.arange(len(weekly_gini)))
    return mean_gini + 88.0 * min(0, slope) - 0.5 * residuals.std(), mean_gini, slope, residuals.std(), weekly_gini

## 6. Time-based split + train

In [ ]:
val_mask = week_num > TRAIN_CUTOFF

X_train, y_train = X[train_mask], y[train_mask]
X_val,   y_val   = X[val_mask],   y[val_mask]
week_val = week_num[val_mask]

print(f'Train: {X_train.shape[0]:,} ({y_train.mean():.3%} pos)')
print(f'Val:   {X_val.shape[0]:,} ({y_val.mean():.3%} pos)')

params = {
    'objective': 'binary', 'metric': 'auc',
    'n_estimators': 1000, 'learning_rate': 0.05,
    'num_leaves': 63, 'min_child_samples': 50,
    'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 1,
    'reg_alpha': 0.1, 'reg_lambda': 0.1,
    'scale_pos_weight': (y_train == 0).sum() / (y_train == 1).sum(),
    'n_jobs': -1, 'verbose': -1, 'random_state': 42,
}

model = lgb.LGBMClassifier(**params)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)
print(f'Best iteration: {model.best_iteration_}')

## 7. Evaluate

In [ ]:
val_preds = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, val_preds)
score, mean_gini, slope, resid_std, weekly_gini = gini_stability(week_val, y_val, val_preds)

print(f'Validation AUC:        {auc:.4f}')
print(f'Gini stability score:  {score:.4f}')
print(f'  Mean weekly Gini:    {mean_gini:.4f}')
print(f'  Trend slope:         {slope:.6f}')
print(f'  Residual std:        {resid_std:.4f}')

valid_weeks = [w for w in np.unique(week_val) if y_val[week_val == w].sum() > 0]
plt.figure(figsize=(10, 4))
plt.plot(valid_weeks, weekly_gini, marker='o', linewidth=1.5)
plt.axhline(mean_gini, color='red', linestyle='--', label=f'Mean={mean_gini:.3f}')
plt.xlabel('WEEK_NUM'); plt.ylabel('Gini')
plt.title('Per-week Gini — Submission 2')
plt.legend(); plt.tight_layout(); plt.show()

## 8. Feature importance

In [ ]:
importance = pl.DataFrame({
    'feature': stable_features,
    'importance': model.feature_importances_
}).sort('importance', descending=True)
print(importance.head(20))

top20 = importance.head(20)
plt.figure(figsize=(10, 8))
plt.barh(top20['feature'].to_list()[::-1], top20['importance'].to_list()[::-1])
plt.xlabel('Importance'); plt.title('Top 20 features — Submission 2')
plt.tight_layout(); plt.show()

## 9. Retrain on full data & submit

In [ ]:
final_model = lgb.LGBMClassifier(**{**params, 'n_estimators': model.best_iteration_})
final_model.fit(X, y)

submission = pl.DataFrame({
    'case_id': test_base['case_id'],
    'score': final_model.predict_proba(X_test)[:, 1]
})
submission.write_csv(f'{OUTPUT}/submission.csv')
print(f'Saved {len(submission):,} predictions')
print(submission.describe())